In [ ]:
# Set run type - Allows user to run code in DEMO state (using dummy dataset) or Full Version, accessing resources from Google Colab
# Set 'ON' to run Demo version, 'OFF' to run Full Version (need access to Google Colab resources)
import os
os.environ['RUN_DEMO'] = 'ON' 

## Libraries and Imports

In [ ]:
import src.config as c
import src.load_data as ld
viirs_files = ld.get_filepaths('VIIRS')
viirs_to_load = ld.to_load_viirs(viirs_files,[2018])
viirs_to_load

In [ ]:
import src.load_data as ld

In [ ]:
# import sys
# import os

# # Set working directory and mount GoogleDrive
# drive.mount('/content/drive')
# wd = '/content/drive/MyDrive/Colab Notebooks/WildFirePrediction/[2]ProgramWildFirePredict'
# if os.path.exists(wd):
#   print(f"✅  Success - working directory (wd) set as [{wd}]")
# else:
#   print(f"❌ Error: Could not set [{wd}] as root directory for project. Review path and try again...")
# sys.path.append(wd)

In [ ]:
import pandas as pd

d1 = {'lon':     [1,            2,             3,           4,            5],
      'lat':     [10,           11             ,12,         13,           14],
      'date':    ['2024-01-01', '2024-02-02', '2024-03-03', '2024-04-04', '2024-05-05'],
      'another': [0.1,          0.2,           0.3,         0.4,           0.5]}

d2 = {'lon':     [1,            3,            9,            4,            5],
      'lat':     [10,           11,           12,           20,           14],
      'date':    ['2024-01-01', '2024-02-02', '2025-03-03', '2024-04-04', '2024-05-05'],
      'another': [0.1,          0.2,           0.3,          0.4,          0.101]}
                 
de = {'lon':     [1,            2,             3,           4,            5,             3,           9,            4          ],
      'lat':     [10,           11             ,12,         13,           14,            11,          12,           20         ],
      'date':    ['2024-01-01', '2024-02-02', '2024-03-03', '2024-04-04', '2024-05-05',  '2024-02-02','2025-03-03', '2024-04-04'],
      'another': [0.1,          0.2,           0.3,         0.4,           0.5,          0.2,          0.3,          0.4       ]}

cols_merge = ['lon','lat','date']
df1 = pd.DataFrame(d1)
df2 = pd.DataFrame(d2)

df_diff = df2.loc[ ~df2.set_index(cols_merge).index.isin(df1.set_index(cols_merge).index)]
df_diff

df_merged = pd.concat([df1,df_diff], ignore_index=True)
df_merged.shape[0]


In [1]:
import os
os.environ.setdefault("RUN_DEMO", "ON")
import src.config as c
import src.load_data as ld
import geopandas as gpd
from pathlib import Path
import matplotlib.pyplot as plt
import utils as u

# --------------------------
# VARIABLES
# --------------------------
YEAR_FILTER = [2019]
CRS = "EPSG: 4326"          # Set Coordinate Reference System (CRS) so it is uniform across all data inputs
SATELITE_IMAGES = "COPERNICUS/S2_SR_HARMONIZED"         
# --------------------------
# VIIRS DATA
# --------------------------
viirs_dict = ld.viirs_load_pipeline(dir_name = 'VIIRS',
                                    crs = CRS,
                                    date_range = YEAR_FILTER)
df_viirs = viirs_dict.get('df_viirs')
print(f"{'='*80}")
print(f"VIIRS Data")
print(f"\tData Type: {type(df_viirs)}")
print(f"\t📅 Date Range: {df_viirs['acq_date'].min()} to {df_viirs['acq_date'].max()}")


# --------------------------
# UK GRID 
# --------------------------
df_uk_grid = ld.load_uk_grid(file_name='ukcp18-uk-land-12km.shp',
                             crs=CRS)
print(f"{'='*80}")
print(f"UK Grid")
print(f"Shape: {df_uk_grid.shape}")

# Grids by Day
print(f"{'='*80}")
print(f"UK Grid Daily")
dates = u.extract_year_range(df_viirs)
df_daily_grid = df_uk_grid.copy()
df_daily_grid['join_key'] = 1
df_daily_grid = df_daily_grid.merge(dates, on='join_key').drop(columns='join_key')
print(f"Shape: {df_daily_grid.shape}")
print(f"Columns: {df_daily_grid.columns}")



VIIRS Data
	Data Type: <class 'geopandas.geodataframe.GeoDataFrame'>
	📅 Date Range: 2019-01-01 to 2019-12-31
UK Grid
Shape: (1692, 4)
UK Grid Daily
Shape: (617580, 5)
Columns: Index(['id', 'x_coord', 'y_coord', 'geometry', 'date'], dtype='object')


In [2]:
uk_grid_sample = df_daily_grid[df_daily_grid['date'] == '2019-01-01']
#uk_grid_sample.explore()

In [3]:
import ee
SATELITE_IMAGES = "COPERNICUS/S2_SR_HARMONIZED"         

try: 
    ee.Initialize(project="ee-enmanuelmorego")
except:
    ee.Authenticate
    ee.Initialize(project="ee-enmanuelmorego")

s2 = ee.ImageCollection(SATELITE_IMAGES)

In [ ]:
uk_grid_sample

In [ ]:
# X = Longitude, Y = Latitude
import pandas as pd
df_s2 = pd.DataFrame()
for i, r in uk_grid_sample.iterrows():
    date                = r['date']
    date_value          = date.strftime("%Y-%m-%d")
    polygon_value       = r['geometry']

    s2_filter_date      = s2.filterDate(date_value, date_value)
    s2_geom_loc         = ee.Geometry.Polygon(polygon_value)
    s2_filter_location  = s2_filter_date.filterBounds(s2_geom_loc)
    s2_img_data         = s2_filter_location.first()    
    df_s2               = pd.concat([df_s2,
                                     pd.DataFrame({'date': date, 
                                                   'geometry': polygon_value, 
                                                   's2_metadata': s2_img_data})
                                     ])

df_s2.head()

In [ ]:
def geodf_to_ee(geo_df: gpd.GeoDataFrame) -> ee.FeatureCollection:
    """
    Convert GeoPandasDataFrame to a GoogleEarth Feature Collection to extract Sentinel-2 images

    Args:
        geo_df (GeoDataFrame): Data frame containing the plygon for the UK Grids, for each date of the date range

    Returns:
        Feature Collection
    """
    features = []
    for _, r in geo_df.iterrows():
        date_f  = r["date"].strftime("%Y-%m-%d")
        geom    = ee.Geometry.Polygon(list(r.geometry.exterior.coords))
        feature = ee.Feature(geom,
                             {'date': date_f})
        features.append(feature)
    return ee.FeatureCollection(features)

def attach_s2_metadata(feature: ee.FeatureCollection):
    """
    Extract the metadata for a given feature from FeatureCollection (date, polygon)
    - Filter Sentinel-2 image
    - Select best available image
    - Attach metadada
    """
    date = ee.Date(feature.get("date"))
    geom = feature.geometry()

    # Get images for the day
    s2_day      = (s2
                   .filterDate(date, date.advance(1, "day"))
                   .filterBounds(geom)
                   .sort("CLOUDY_PIXEL_PERCENTAGE"))
    # Select best image
    img         = s2_day.first()
    # Extract metadata
    s2_metadata = feature.set({'s2_id':     ee.Algorithms.If(img, img.get("system:id"), None),
                               "could_pct": ee.Algorithms.If(img, img.get("CLOUDY_PIXEL_PERCENTAGE"), None)
                               }) 
    return s2_metadata

def s2_to_pandas(fc: ee.FeatureCollection) -> 

In [ ]:
f_collection = geodf_to_ee(uk_grid_sample)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80)))
s2_result = attach_s2_metadata(s2)

/Users/enmanuelmoreno/.local/share/virtualenvs/MScProjectWildFirePredict-q-5Auf5i/lib/python3.12/site-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for COPERNICUS/S2_SR! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by COPERNICUS/S2_SR_HARMONIZED

Learn more: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR

  warnings.warn(warning, category=DeprecationWarning)
